# Specialiserede Modeller — 2: NN-udbygninger — overfitting-værktøjskassen

I Intro-ML mødte I overfitting som en fjern trussel ("modellen lærer udenad"). I denne
notebook skal I se den ske LIVE — og lære de fire klassiske våben imod den:
**valideringssæt & early stopping, dropout, weight decay og batch normalization**.
Det er de udbygninger, stort set alle rigtige neurale netværk trænes med.

Dagens data er alvorlige: **918 patienter** — kan vi forudsige hjertesygdom?
Små, virkelige, støjede sundhedsdata er PRÆCIS der, hvor overfitting bider.

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
# Henter hjerte-data fra GitHub (Plan B: upload heart.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/data/heart.csv

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("heart.csv")
df.head()

Hver række er en patient: alder, køn, blodtryk, kolesterol, maks-puls, brystsmerte-type
m.m. — og `HeartDisease` (1 = hjertesygdom). Tekstkolonnerne one-hot-koder vi med
`pd.get_dummies` (hver kategori får sin egen 0/1-kolonne), og så standardiserer vi,
som vi plejer:

In [ ]:
X = pd.get_dummies(df.drop(columns=["HeartDisease"])).astype("float32")
print("features efter one-hot:", X.shape)

X_np = X.values
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)     # standardisering, som altid
y_np = df["HeartDisease"].values.astype("float32")
print("andel med hjertesygdom:", round(float(y_np.mean()), 3))

# 3: Valideringssættet — den manglende brik

I Intro-ML delte vi data i **train** og **test**. Men der er et hul i den plan: når I
skruer på epoker, læringsrate og netstørrelse og måler på testsættet hver gang — så
vælger I til sidst præcis de indstillinger, der tilfældigvis passer TESTSÆTTET. I har
"slidt eksamen op": modellen består en prøve, den reelt har fået lov at øve sig på.

Derfor deler professionelle ALTID i tre:

| Sæt | Rolle | Skole-analogi |
|---|---|---|
| **Træning** (60 %) | modellen lærer af det | lektier |
| **Validering** (20 %) | VI vælger indstillinger efter det | terminsprøver |
| **Test** (20 %) | røres ÉN gang, til allersidst | eksamen |

Splittet laver vi med `train_test_split` to gange — først 60/40, så deles de 40 i to:

In [ ]:
X_traen_np, X_rest, y_traen_np, y_rest = train_test_split(X_np, y_np, test_size=0.4, random_state=42)
X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

X_traen, X_val, X_test = map(torch.tensor, (X_traen_np, X_val_np, X_test_np))
y_traen, y_val, y_test = map(torch.tensor, (y_traen_np, y_val_np, y_test_np))

print("træning:", len(X_traen), "| validering:", len(X_val), "| test:", len(X_test))

## Fremprovokér overfitting!

Nu gør vi bevidst alt det, man ikke må: et KÆMPE netværk (over 70.000 parametre) på
BITTESMÅ data (550 patienter), trænet længe. Undervejs noterer vi tabet på både
træningsdata og valideringsdata — valideringstabet beregnes i `torch.no_grad()`, for
modellen må gerne blive MÅLT på valideringssættet, men aldrig LÆRE af det:

In [ ]:
def byg_net():
    return nn.Sequential(
        nn.Linear(X.shape[1], 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 1), nn.Sigmoid())

model = byg_net()
print("parametre:", sum(p.numel() for p in model.parameters()))

tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
traen_tab, val_tab = [], []

for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()

    with torch.no_grad():                          # mål, men lær ikke!
        traen_tab.append(tab.item())
        val_tab.append(tabsfunktion(model(X_val).squeeze(), y_val).item())

In [ ]:
plt.plot(traen_tab, label="træningstab")
plt.plot(val_tab, label="valideringstab")
plt.xlabel("epoke")
plt.ylabel("BCE-tab")
plt.legend()
plt.title("Den vigtigste figur i praktisk deep learning")
plt.show()

print(f"laveste valideringstab: {min(val_tab):.3f} ved epoke {val_tab.index(min(val_tab))}")

**DÉR er overfitting.** Træningstabet falder og falder (modellen lærer patienterne
udenad) — men valideringstabet bunder omkring epoke ~130 og **stiger derefter**. Fra
det punkt bliver modellen bedre til at HUSKE og dårligere til at FORUDSIGE. Al træning
efter bunden er ikke bare spildt — den er skadelig.

## Early stopping: stop, mens legen er god

Det oplagte modtræk: gem modellen, hver gang valideringstabet sætter ny bundrekord —
og brug til sidst den gemte udgave i stedet for den sidste:

In [ ]:
torch.manual_seed(42)
model = byg_net()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

bedste_val_tab = 999.0
bedste_epoke = 0
bedste_vaegte = None

for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()

    with torch.no_grad():
        val_nu = tabsfunktion(model(X_val).squeeze(), y_val).item()
    if val_nu < bedste_val_tab:                              # ny bundrekord?
        bedste_val_tab = val_nu
        bedste_epoke = epoke
        bedste_vaegte = copy.deepcopy(model.state_dict())    # gem en KOPI af vægtene

model.load_state_dict(bedste_vaegte)                         # spol tilbage til rekorden
print(f"bedste model: epoke {bedste_epoke} (valideringstab {bedste_val_tab:.3f})")

with torch.no_grad():
    val_acc = ((model(X_val).squeeze() > 0.5).float() == y_val).float().mean()
print(f"validerings-accuracy: {val_acc.item():.1%}")

(`model.state_dict()` er en ordbog med alle modellens vægte, og `copy.deepcopy` tager
en RIGTIG kopi — ellers ville vores "gemte" vægte bare pege på de levende vægte, der
ændrer sig videre. `load_state_dict` lægger dem tilbage.)

### Opgaver

##### Opgave 3.1
Aflæs U-kurven fra det store eksperiment: cirka ved hvilken epoke begynder
valideringstabet at stige? Og hvad er der galt med at sige *"men træningstabet falder
jo stadig — modellen bliver da bedre!"*?

*Skriv jeres svar her:* $\dots$

##### Opgave 3.2
Hvorfor kan vi ikke bare bruge TESTSÆTTET til at vælge antal epoker (og droppe
valideringssættet)? Hvad ville der ske med troværdigheden af vores allersidste
test-måling?

*Skriv jeres svar her:* $\dots$

##### Opgave 3.3
Udfyld de to `test_size`-værdier, så splittet bliver præcis 60/20/20. (Tænk: andet
split deler RESTEN — hvor stor en andel af de 40 % skal blive til test?)

In [ ]:
X_a, X_r, y_a, y_r = train_test_split(X_np, y_np, test_size=..., random_state=42)
X_v, X_t, y_v, y_t = train_test_split(X_r, y_r, test_size=..., random_state=42)
print(len(X_a), len(X_v), len(X_t))   # skal give ca. 550 / 184 / 184

##### Opgave 3.4
Gentag overfitting-eksperimentet med et LILLE net: skift alle 256-taller ud med 16.
Hvordan ser kurverne ud nu — og hvorfor giver det mening, at et lille net overfitter
mindre?

In [ ]:
torch.manual_seed(42)
model_lille = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(),     # ← 256 → 16 alle tre steder
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())
optimizer = torch.optim.Adam(model_lille.parameters(), lr=0.0001)
traen_tab2, val_tab2 = [], []
for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model_lille(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()
    with torch.no_grad():
        traen_tab2.append(tab.item())
        val_tab2.append(tabsfunktion(model_lille(X_val).squeeze(), y_val).item())

plt.plot(traen_tab2, label="træningstab")
plt.plot(val_tab2, label="valideringstab")
plt.legend()
plt.show()

##### Opgave 3.5 (find fejlen)
Denne kammerats "valideringskurve" ser mistænkeligt godt ud — den falder bare og
falder, intet U! Kig godt på linjen, hvor valideringstabet beregnes. Hvad er der sket —
og hvorfor er netop DEN fejl så farlig? (Den giver jo ingen fejlbesked...)

In [ ]:
torch.manual_seed(42)
model_k = byg_net()
optimizer = torch.optim.Adam(model_k.parameters(), lr=0.0001)
val_tab_k = []
for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model_k(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()
    with torch.no_grad():
        val_tab_k.append(tabsfunktion(model_k(X_traen).squeeze(), y_traen).item())

plt.plot(val_tab_k, label="'valideringstab'")
plt.legend()
plt.show()

##### Opgave 3.6
Overfitting elsker små datasæt. Brug kun de første 275 træningspatienter
(`X_traen[:275]`, `y_traen[:275]`) og kør det store net igen. Kommer U-kurvens bund
tidligere eller senere — og bliver gabet mellem kurverne større eller mindre?

In [ ]:
torch.manual_seed(42)
model_h = byg_net()
optimizer = torch.optim.Adam(model_h.parameters(), lr=0.0001)
traen_tab3, val_tab3 = [], []
for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model_h(X_traen).squeeze(), y_traen)      # ← brug kun de første 275!
    tab.backward()
    optimizer.step()
    with torch.no_grad():
        traen_tab3.append(tab.item())
        val_tab3.append(tabsfunktion(model_h(X_val).squeeze(), y_val).item())

plt.plot(traen_tab3, label="træningstab")
plt.plot(val_tab3, label="valideringstab")
plt.legend()
plt.show()

##### Opgave 3.7
Udfyld de tre manglende linjer i early stopping (gem rekorden, epoken og en KOPI af
vægtene).

In [ ]:
torch.manual_seed(42)
model_es = byg_net()
optimizer = torch.optim.Adam(model_es.parameters(), lr=0.0001)
bedste_val_tab = 999.0
bedste_epoke = 0
bedste_vaegte = None

for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model_es(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()
    with torch.no_grad():
        val_nu = tabsfunktion(model_es(X_val).squeeze(), y_val).item()
    if val_nu < bedste_val_tab:
        bedste_val_tab = ...
        bedste_epoke = ...
        bedste_vaegte = ...

model_es.load_state_dict(bedste_vaegte)
print(f"bedste epoke: {bedste_epoke} (val-tab {bedste_val_tab:.3f})")

##### Opgave 3.8 (ekstra)
Rigtig early stopping venter ikke til epoke 600 — den AFBRYDER, når valideringstabet
ikke har sat bundrekord i `taalmodighed` epoker i træk. Udfyld tæller-logikken
(nulstil ved rekord, tæl op ellers). Hvor mange epoker sparede I?

In [ ]:
torch.manual_seed(42)
model_p = byg_net()
optimizer = torch.optim.Adam(model_p.parameters(), lr=0.0001)
bedste_val_tab = 999.0
taalmodighed = 50
siden_rekord = 0

for epoke in range(600):
    optimizer.zero_grad()
    tab = tabsfunktion(model_p(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()
    with torch.no_grad():
        val_nu = tabsfunktion(model_p(X_val).squeeze(), y_val).item()
    if val_nu < bedste_val_tab:
        bedste_val_tab = val_nu
        siden_rekord = ...          # rekord! nulstil tælleren
    else:
        siden_rekord = ...          # ingen rekord — tæl op
    if siden_rekord >= taalmodighed:
        print(f"early stopping ved epoke {epoke}!")
        break

# 4: Værktøjskassen — dropout, weight decay & batch norm

Early stopping behandler symptomet (stop før udenadslæren). De næste værktøjer angriber
årsagen: de gør det SVÆRERE for nettet at lære udenad.

For ikke at skrive det samme loop fem gange pakker vi det i en funktion — læs den, det
er jeres velkendte rytme plus én ny detalje: **`model.train()` og `model.eval()`**.
De to kald tænder og slukker for "træningsopførsel". Hvorfor det pludselig er vigtigt,
ser I om lidt:

In [ ]:
def traen_med_kurver(model, epoker=600, lr=0.0001, weight_decay=0.0):
    tabsfunktion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    traen_tab, val_tab = [], []
    for epoke in range(epoker):
        model.train()                                # træningsopførsel TIL
        optimizer.zero_grad()
        tab = tabsfunktion(model(X_traen).squeeze(), y_traen)
        tab.backward()
        optimizer.step()

        model.eval()                                 # træningsopførsel FRA, før vi måler
        with torch.no_grad():
            traen_tab.append(tabsfunktion(model(X_traen).squeeze(), y_traen).item())
            val_tab.append(tabsfunktion(model(X_val).squeeze(), y_val).item())
    return traen_tab, val_tab


def val_accuracy(model):
    model.eval()
    with torch.no_grad():
        return ((model(X_val).squeeze() > 0.5).float() == y_val).float().mean().item()

## Dropout: træn med huller i holdet

**Dropout** slukker i hvert træningsskridt en tilfældig andel af neuronerne (fx 50 %).
Så kan ingen enkelt neuron få lov at huske hele facitlisten — holdet tvinges til at
sprede viden ud. Som fodboldtræning, hvor tilfældige holdkammerater udebliver hver
gang: alle ender med at kunne alle positioner.

Vigtigt: hullerne er KUN til træning. Når modellen skal bruges, spiller hele holdet —
det er derfor `model.eval()` findes. I PyTorch er dropout bare et lag:

In [ ]:
torch.manual_seed(42)
model_dropout = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())

traen_tab_d, val_tab_d = traen_med_kurver(model_dropout)

plt.plot(val_tab, label="uden dropout (fra afsnit 3)")
plt.plot(val_tab_d, label="med dropout 0.5")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()
print(f"validerings-accuracy med dropout: {val_accuracy(model_dropout):.1%}")

## Weight decay: hold vægtene små

Udenadslære kræver typisk store, ekstreme vægte ("HVIS præcis denne kombination, SÅ
syg!"). **Weight decay** lægger en lille straf på store vægte oveni tabet, så nettet
foretrækker små vægte — og dermed enkle, glatte løsninger. I PyTorch er det
bogstaveligt talt ét ekstra argument til optimizeren:

In [ ]:
torch.manual_seed(42)
model_wd = byg_net()
traen_tab_w, val_tab_w = traen_med_kurver(model_wd, weight_decay=0.01)

plt.plot(val_tab, label="uden noget")
plt.plot(val_tab_w, label="med weight decay 0.01")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()
print(f"validerings-accuracy med weight decay: {val_accuracy(model_wd):.1%}")

## Batch normalization: stabilisatoren

**Batch norm** standardiserer tallene MELLEM lagene (jeres standardiserings-trick,
bare gentaget inde i nettet). Det gør træning af DYBE netværk markant mere stabil og
hurtig, og næsten alle store moderne net bruger det.

Men lad os være ærlige og AFPRØVE det i stedet for at tro på reklamen:

In [ ]:
torch.manual_seed(42)
model_bn = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())

traen_tab_b, val_tab_b = traen_med_kurver(model_bn)

fig, akser = plt.subplots(1, 2, figsize=(12, 4))
akser[0].plot(traen_tab, label="uden")
akser[0].plot(traen_tab_b, label="med batch norm")
akser[0].set_title("træningstab")
akser[0].legend()
akser[1].plot(val_tab, label="uden")
akser[1].plot(val_tab_b, label="med batch norm")
akser[1].set_title("valideringstab")
akser[1].legend()
plt.show()

Se godt efter: med batch norm **styrtdykker træningstabet** (stabilisatoren virker!)
— men valideringstabet vender TIDLIGERE og stiger STEJLERE. På et lillebitte datasæt
som vores accelererer batch norm bare udenadslæren. Lærdommen er vigtig: batch norm er
et værktøj til at få STORE, DYBE net til at træne stabilt — det er ikke et middel mod
overfitting. Kend dit værktøjs job.

## Den samlede opskrift

I praksis kombinerer man: et fornuftigt net + dropout + lidt weight decay + early
stopping. Lad os køre hele pakken — og til allersidst, én gang, røre testsættet:

In [ ]:
torch.manual_seed(42)
model_fuld = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())

tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model_fuld.parameters(), lr=0.0001, weight_decay=0.01)
bedste_val_tab = 999.0
bedste_vaegte = None

for epoke in range(600):
    model_fuld.train()
    optimizer.zero_grad()
    tab = tabsfunktion(model_fuld(X_traen).squeeze(), y_traen)
    tab.backward()
    optimizer.step()

    model_fuld.eval()
    with torch.no_grad():
        val_nu = tabsfunktion(model_fuld(X_val).squeeze(), y_val).item()
    if val_nu < bedste_val_tab:
        bedste_val_tab = val_nu
        bedste_vaegte = copy.deepcopy(model_fuld.state_dict())

model_fuld.load_state_dict(bedste_vaegte)
model_fuld.eval()
with torch.no_grad():
    test_acc = ((model_fuld(X_test).squeeze() > 0.5).float() == y_test).float().mean()
print(f"ENDELIG test-accuracy (målt én gang!): {test_acc.item():.1%}")

| Værktøj | Hvad gør det? | Hvornår? |
|---|---|---|
| Valideringssæt | ærlig måling undervejs | ALTID |
| Early stopping | stop før udenadslæren | næsten altid |
| Dropout | sluk neuroner → tving samarbejde | mellemstore/store net |
| Weight decay | straf store vægte → enkle løsninger | næsten altid (små doser) |
| Batch norm | stabilisér træning af DYBE net | dybe net, store datasæt |
| Mindre net / mere data | angrib årsagen direkte | når muligt! |

### Opgaver

##### Opgave 4.1
Prøv dropout på **0.1, 0.5 og 0.9** og sammenlign valideringskurverne i ét plot. Hvad
går der galt ved 0.9 — og hvad hedder det modsatte af overfitting?

In [ ]:
for rate in [0.1]:      # ← prøv [0.1, 0.5, 0.9]
    torch.manual_seed(42)
    m = nn.Sequential(
        nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(rate),
        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(rate),
        nn.Linear(256, 1), nn.Sigmoid())
    t_tab, v_tab = traen_med_kurver(m)
    plt.plot(v_tab, label=f"dropout {rate}")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()

##### Opgave 4.2 (find fejlen)
En kammerat måler accuracy på dropout-modellen — og får et NYT tal, hver gang cellen
køres?! Og tallene er lavere end forventet. Kør cellen 3-4 gange og se selv. Én linje
er forkert — hvilken, og hvorfor ændrer den alt?

In [ ]:
model_dropout.train()   # ... den har vel ikke noget at sige?
with torch.no_grad():
    acc = ((model_dropout(X_val).squeeze() > 0.5).float() == y_val).float().mean()
print(f"accuracy: {acc.item():.1%}   — kør cellen igen: nyt tal!")

##### Opgave 4.3
Udfyld `weight_decay`-argumentet (prøv 0.01), og tjek at valideringskurven bliver
fladere end uden.

In [ ]:
torch.manual_seed(42)
m = byg_net()
optimizer = torch.optim.Adam(m.parameters(), lr=0.0001, ...)
# genbrug evt. traen_med_kurver i stedet — den tager weight_decay som parameter
t_tab, v_tab = traen_med_kurver(m, weight_decay=...)
plt.plot(val_tab, label="uden")
plt.plot(v_tab, label="med weight decay")
plt.legend()
plt.show()

##### Opgave 4.4
Weight decay har også en overdosis: prøv **0, 0.0001, 0.01 og 1.0** og sammenlign
kurver/accuracy. Hvad sker der ved 1.0, og hvorfor?

In [ ]:
for wd in [0.0]:      # ← prøv [0, 0.0001, 0.01, 1.0]
    torch.manual_seed(42)
    m = byg_net()
    t_tab, v_tab = traen_med_kurver(m, weight_decay=wd)
    plt.plot(v_tab, label=f"wd = {wd}")
plt.legend()
plt.show()

##### Opgave 4.5
Modellen skal bruges på rigtige patienter. Diskutér: hvad koster en **falsk negativ**
(syg patient, modellen siger rask) i forhold til en **falsk positiv** (rask patient,
modellen siger syg)? Og hvad kunne man gøre ved 0,5-grænsen, hvis man hellere vil have
den ene slags fejl end den anden?

*Skriv jeres svar her:* $\dots$

##### Opgave 4.6
Byg jeres bedste opskrift: kombinér valgfrit netværksstørrelse, dropout, weight decay
og early stopping — men mål KUN på valideringssættet, mens I eksperimenterer. Når I har
valgt jeres endelige opskrift: mål på testsættet ÉN gang. Hvor tæt ligger jeres
validerings- og test-accuracy?

In [ ]:
# eksperimentér her (mål kun på validering!) — og til allersidst: én test-måling
torch.manual_seed(42)
m = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())
t_tab, v_tab = traen_med_kurver(m, weight_decay=0.01)
print(f"validering: {val_accuracy(m):.1%}")

# NÅR (og kun når) I er færdige med at eksperimentere:
# m.eval()
# with torch.no_grad():
#     test_acc = ((m(X_test).squeeze() > 0.5).float() == y_test).float().mean()
# print(f"test: {test_acc.item():.1%}")

##### Opgave 4.7 (ekstra)
Batch norm fik hårde ord ovenfor — men den lovede også HURTIGERE træning. Mål det:
hvor mange epoker tager det henholdsvis med og uden batch norm at få træningstabet
under 0.3? (Skriv en for-løkke, der finder den første epoke under grænsen.)

In [ ]:
torch.manual_seed(42)
m_uden = byg_net()
t_uden, v_uden = traen_med_kurver(m_uden)

torch.manual_seed(42)
m_med = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())
t_med, v_med = traen_med_kurver(m_med)

# find første epoke hvor tabet er under 0.3 — for begge:
...

##### Opgave 4.8 (ekstra)
Tilbage til Intro-ML-tricket: gennemsnittet af en True/False-række. Beregn hvor mange af
testsættets FAKTISK SYGE patienter jeres endelige model fanger (recall) — og sammenlign
med den samlede accuracy. Hvilket af de to tal ville I vise til en læge først?

In [ ]:
model_fuld.eval()
with torch.no_grad():
    gaet = (model_fuld(X_test).squeeze() > 0.5).float()

syge = y_test == 1
recall = ...          # andelen af de syge, som modellen fanger
print(f"accuracy: {(gaet == y_test).float().mean().item():.1%}")
print(f"recall:   {recall:.1%}")

# Godt gået!

I har nu den professionelle værktøjskasse: train/val/test-splittet, U-kurven, early
stopping med vægt-kopier, dropout (og eval()-reglen!), weight decay og en ærlig
forståelse af batch norm. Det her er forskellen på legetøjs-træning og rigtig træning
— ALT hvad I træner fremover, bør bruge validering + early stopping som minimum.